# GCI In-Class Competition: Home Credit Default Risk
## Notebook 1: Advanced Feature Engineering

**Objective**: Create rich feature set for LightGBM, XGBoost, CatBoost ensemble
**Target**: ROC-AUC ≈ 0.762 on 5-fold CV (the 33-column subset caps what is achievable; v4 scored 0.76244)
**Hardware**: CPU is sufficient (all final members are CPU-trained)

> **Status:** this notebook builds the 101-column base matrix. The final pipeline applies an
> E2 layer (15 enrichments, 18 dead-column drops) to reach the 98 columns used by v4
> (`PROCESS_LOG.md`); notebooks 02–04 are historical.

---

### Feature Engineering Strategy
1. **Missing Value Indicators** - Binary flags for high-missing columns
2. **Domain Financial Ratios** - Credit/income/annuity relationships
3. **EXT_SOURCE Interactions** - Strongest predictors, pairwise combinations
4. **Categorical Encoding** - Target encoding (CV-only) + Frequency encoding
5. **Binned/Quantile Features** - Capture non-linear relationships
6. **Aggregation Features** - Bureau query summaries, EXT_SOURCE stats
7. **Anomaly Handling** - DAYS_EMPLOYED = 365243 → NaN + indicator

⚠️ **CRITICAL**: All target encoding done WITHIN CV folds to prevent leakage!

In [ ]:
# 1. Setup and Library Imports
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import time
import os
import gc
import json
import warnings
warnings.filterwarnings('ignore')

print(f"LightGBM: {lgb.__version__}")
print(f"XGBoost: {xgb.__version__}")
print(f"CatBoost: {cb.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

In [ ]:
# 2. Google Drive Mount (for persistence across Colab sessions)
from google.colab import drive
drive.mount('/content/drive')

# Set working directory on Drive
WORK_DIR = '/content/drive/MyDrive/gci_competition'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/features', exist_ok=True)
os.makedirs(f'{WORK_DIR}/models', exist_ok=True)
os.makedirs(f'{WORK_DIR}/optuna_studies', exist_ok=True)
os.makedirs(f'{WORK_DIR}/submissions', exist_ok=True)

print(f"Working directory: {WORK_DIR}")
print("Folders created successfully!")

In [ ]:
# 3. Load Data
# Update these paths to match your Drive structure
TRAIN_PATH = f'{WORK_DIR}/input/train.csv'
TEST_PATH = f'{WORK_DIR}/input/test.csv'
SAMPLE_SUB_PATH = f'{WORK_DIR}/input/sample_submission.csv'

# Fallback: if files not on Drive, check local
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = 'train.csv'
    TEST_PATH = 'test.csv'
    SAMPLE_SUB_PATH = 'sample_submission.csv'

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Sample submission shape: {sample_sub.shape}")
print(f"\nTarget distribution:")
print(train['TARGET'].value_counts())
print(f"Positive class rate: {train['TARGET'].mean():.4f}")

In [ ]:
# 4. Initial EDA - Missing Values & Column Types
print("=== Missing Values (Train) ===")
missing_train = train.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)
print(missing_train)
print(f"\nTotal columns with missing: {len(missing_train)}")

print("\n=== Column Types ===")
print(train.dtypes.value_counts())

print("\n=== Categorical Columns ===")
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
print(f"Count: {len(cat_cols)}")
for c in cat_cols:
    print(f"  {c}: {train[c].nunique()} unique values")

print("\n=== Numerical Columns ===")
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
print(f"Count: {len(num_cols)}")

In [ ]:
# 5. Feature Engineering Functions (Modular, Reusable)

def add_missing_indicators(df, missing_cols):
    """Add binary indicator columns for missing values."""
    for col in missing_cols:
        if col in df.columns:
            df[f'{col}_WAS_MISSING'] = df[col].isnull().astype(np.int8)
    return df

def add_domain_ratios(df):
    """Add domain-specific financial ratios."""
    eps = 1e-5
    
    # Existing ratios from baseline
    df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + eps)
    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + eps)
    df['CREDIT_TERM'] = df['AMT_ANNUITY'] / (df['AMT_CREDIT'] + eps)
    df['DAYS_EMPLOYED_RATIO'] = df['DAYS_EMPLOYED'] / (df['DAYS_BIRTH'] + eps)
    df['GOODS_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / (df['AMT_CREDIT'] + eps)
    df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + eps)
    
    # NEW: Additional domain ratios
    df['CREDIT_TO_GOODS_RATIO'] = df['AMT_CREDIT'] / (df['AMT_GOODS_PRICE'] + eps)
    df['ANNUITY_TO_CREDIT_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_CREDIT'] + eps)
    df['INCOME_TO_CREDIT_RATIO'] = df['AMT_INCOME_TOTAL'] / (df['AMT_CREDIT'] + eps)
    df['PHONE_CHANGE_TO_AGE'] = df['DAYS_LAST_PHONE_CHANGE'] / (df['DAYS_BIRTH'] + eps)
    df['EMPLOYED_TO_BIRTH_RATIO'] = df['DAYS_EMPLOYED'] / (df['DAYS_BIRTH'] + eps)
    df['REGISTRATION_TO_BIRTH'] = df['DAYS_REGISTRATION'] / (df['DAYS_BIRTH'] + eps)
    df['ID_PUBLISH_TO_BIRTH'] = df['DAYS_ID_PUBLISH'] / (df['DAYS_BIRTH'] + eps)
    
    # Car age ratios
    df['CAR_AGE_TO_EMPLOYED'] = df['OWN_CAR_AGE'] / (df['DAYS_EMPLOYED'].abs() + eps)
    df['CAR_AGE_TO_BIRTH'] = df['OWN_CAR_AGE'] / (df['DAYS_BIRTH'].abs() + eps)
    
    return df

def add_ext_source_interactions(df):
    """Add EXT_SOURCE interaction features (strongest predictors)."""
    ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
    
    # Existing from baseline
    df['EXT_SOURCES_MEAN'] = df[ext_cols].mean(axis=1)
    df['EXT_SOURCES_PROD'] = df['EXT_SOURCE_1'].fillna(0) * df['EXT_SOURCE_2'].fillna(0) * df['EXT_SOURCE_3'].fillna(0)
    df['EXT_SOURCES_STD'] = df[ext_cols].std(axis=1)
    df['EXT_SOURCES_MAX'] = df[ext_cols].max(axis=1)
    df['EXT_SOURCES_MIN'] = df[ext_cols].min(axis=1)
    
    # NEW: Pairwise interactions
    df['EXT_1_2_PROD'] = df['EXT_SOURCE_1'].fillna(0) * df['EXT_SOURCE_2'].fillna(0)
    df['EXT_1_3_PROD'] = df['EXT_SOURCE_1'].fillna(0) * df['EXT_SOURCE_3'].fillna(0)
    df['EXT_2_3_PROD'] = df['EXT_SOURCE_2'].fillna(0) * df['EXT_SOURCE_3'].fillna(0)
    
    df['EXT_1_2_DIFF'] = (df['EXT_SOURCE_1'] - df['EXT_SOURCE_2']).abs()
    df['EXT_1_3_DIFF'] = (df['EXT_SOURCE_1'] - df['EXT_SOURCE_3']).abs()
    df['EXT_2_3_DIFF'] = (df['EXT_SOURCE_2'] - df['EXT_SOURCE_3']).abs()
    
    df['EXT_1_2_RATIO'] = df['EXT_SOURCE_1'] / (df['EXT_SOURCE_2'] + 1e-5)
    df['EXT_1_3_RATIO'] = df['EXT_SOURCE_1'] / (df['EXT_SOURCE_3'] + 1e-5)
    df['EXT_2_3_RATIO'] = df['EXT_SOURCE_2'] / (df['EXT_SOURCE_3'] + 1e-5)
    
    # Count of non-null EXT_SOURCE
    df['EXT_SOURCES_COUNT'] = df[ext_cols].notnull().sum(axis=1)
    
    # EXT_SOURCE weighted by availability
    df['EXT_SOURCES_WEIGHTED_MEAN'] = (
        df['EXT_SOURCE_1'].fillna(0) + df['EXT_SOURCE_2'].fillna(0) + df['EXT_SOURCE_3'].fillna(0)
    ) / (df['EXT_SOURCES_COUNT'] + 1e-5)
    
    return df

def add_bureau_aggregations(df):
    """Add aggregations of credit bureau query features."""
    bureau_cols = [
        'AMT_REQ_CREDIT_BUREAU_HOUR',
        'AMT_REQ_CREDIT_BUREAU_MON',
        'AMT_REQ_CREDIT_BUREAU_QRT',
        'AMT_REQ_CREDIT_BUREAU_YEAR'
    ]
    
    df['BUREAU_QUERY_SUM'] = df[bureau_cols].fillna(0).sum(axis=1)
    df['BUREAU_QUERY_MEAN'] = df[bureau_cols].fillna(0).mean(axis=1)
    df['BUREAU_QUERY_MAX'] = df[bureau_cols].fillna(0).max(axis=1)
    df['BUREAU_QUERY_COUNT'] = df[bureau_cols].notnull().sum(axis=1)
    
    # Recent vs historical ratio
    df['BUREAU_RECENT_TO_HIST_RATIO'] = (
        df['AMT_REQ_CREDIT_BUREAU_HOUR'].fillna(0) + df['AMT_REQ_CREDIT_BUREAU_MON'].fillna(0)
    ) / (
        df['AMT_REQ_CREDIT_BUREAU_QRT'].fillna(0) + df['AMT_REQ_CREDIT_BUREAU_YEAR'].fillna(0) + 1e-5
    )
    
    return df

def add_binned_features(df, train_df=None):
    """Add quantile-binned features for key numerical columns."""
    # Key columns to bin
    bin_cols = [
        'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
        'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY',
        'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
        'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE',
        'CNT_FAM_MEMBERS', 'CNT_CHILDREN',
        'REGION_POPULATION_RELATIVE', 'REGION_RATING_CLIENT'
    ]
    
    for col in bin_cols:
        if col in df.columns:
            # Use train quantiles if provided (prevents leakage)
            if train_df is not None and col in train_df.columns:
                try:
                    quantiles = train_df[col].quantile([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]).values
                    df[f'{col}_QBIN'] = pd.cut(df[col], bins=[-np.inf] + list(quantiles) + [np.inf], labels=False, duplicates='drop')
                except:
                    df[f'{col}_QBIN'] = pd.qcut(df[col], q=10, labels=False, duplicates='drop')
            else:
                df[f'{col}_QBIN'] = pd.qcut(df[col], q=10, labels=False, duplicates='drop')
            # Convert to int8
            df[f'{col}_QBIN'] = df[f'{col}_QBIN'].fillna(-1).astype(np.int8)  # missing -> -1
    
    return df

def handle_anomalies(df):
    """Handle known anomalies in the data."""
    # DAYS_EMPLOYED anomaly: 365243 means "not employed" → NaN + indicator
    df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype(np.int8)
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    
    # OWN_CAR_AGE: negative values don't make sense
    if 'OWN_CAR_AGE' in df.columns:
        df['OWN_CAR_AGE'] = df['OWN_CAR_AGE'].clip(lower=0)
    
    return df

print("Feature engineering functions defined successfully!")

In [ ]:
# 6. Apply Feature Engineering

# Combine train and test for consistent feature engineering
y = train['TARGET'].copy()
train_ids = train['SK_ID_CURR'].copy()
test_ids = test['SK_ID_CURR'].copy()

# Drop target and ID for feature engineering
train_fe = train.drop(columns=['TARGET', 'SK_ID_CURR']).copy()
test_fe = test.drop(columns=['SK_ID_CURR']).copy()

# Track original column count
print(f"Original features: {train_fe.shape[1]}")

# Apply feature engineering functions
# 1. Handle anomalies first
train_fe = handle_anomalies(train_fe)
test_fe = handle_anomalies(test_fe)

# 2. Add missing indicators (high-missing columns)
high_missing_cols = [
    'EXT_SOURCE_1', 'OWN_CAR_AGE', 'EXT_SOURCE_3',
    'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_MON',
    'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR',
    'EXT_SOURCE_2', 'CNT_FAM_MEMBERS'
]
train_fe = add_missing_indicators(train_fe, high_missing_cols)
test_fe = add_missing_indicators(test_fe, high_missing_cols)

# 3. Domain ratios
train_fe = add_domain_ratios(train_fe)
test_fe = add_domain_ratios(test_fe)

# 4. EXT_SOURCE interactions
train_fe = add_ext_source_interactions(train_fe)
test_fe = add_ext_source_interactions(test_fe)

# 5. Bureau aggregations
train_fe = add_bureau_aggregations(train_fe)
test_fe = add_bureau_aggregations(test_fe)

# 6. Quantile bins (use train quantiles for both to prevent leakage)
train_fe = add_binned_features(train_fe, train_fe)
test_fe = add_binned_features(test_fe, train_fe)

print(f"\nAfter feature engineering: {train_fe.shape[1]} features")
print(f"New features added: {train_fe.shape[1] - 33}")

In [ ]:
# 7. Categorical Encoding
# Identify categorical columns
cat_cols = train_fe.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")

# For LightGBM: native categorical support (convert to category dtype)
# For XGBoost/CatBoost: need LabelEncoding or CatBoost native

# Store encoders for later use
label_encoders = {}

for col in cat_cols:
    # Work on string copies: the label encoding below overwrites the columns.
    s_tr = train_fe[col].astype(str)
    s_te = test_fe[col].astype(str)
    combined = pd.concat([s_tr, s_te], axis=0)
    
    # Label encode
    le = LabelEncoder()
    le.fit(combined)
    
    train_fe[col] = le.transform(s_tr)
    test_fe[col] = le.transform(s_te)
    
    label_encoders[col] = le
    
    # Frequency encoding: map the STRING values. Mapping the freshly encoded ints
    # against a string-keyed dict produced all-NaN (then -999) in the original run.
    freq_map = combined.value_counts(normalize=True)
    train_fe[f'{col}_FREQ'] = s_tr.map(freq_map).astype(np.float32)
    test_fe[f'{col}_FREQ'] = s_te.map(freq_map).astype(np.float32)

print(f"\nAfter categorical encoding: {train_fe.shape[1]} features")

# Save label encoders for later use
import pickle
with open(f'{WORK_DIR}/features/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
print("Label encoders saved!")

In [ ]:
# 8. Target Encoding (CV-Only - NO LEAKAGE!)
# This function will be called INSIDE each CV fold during training
# We define it here so it's available for notebooks 2 & 3

def get_target_encoding(train_series, test_series, target, n_splits=5, random_state=42, smoothing=10):
    """
    Compute target encoding using CV to prevent leakage.
    Returns encoded train and test series.
    """
    from sklearn.model_selection import StratifiedKFold
    
    # Global mean for smoothing
    global_mean = target.mean()
    
    # Initialize output
    train_encoded = pd.Series(index=train_series.index, dtype=np.float32)
    
    # CV folds
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    for train_idx, val_idx in skf.split(train_series, target):
        # Compute means on training fold only
        train_fold = train_series.iloc[train_idx]
        target_fold = target.iloc[train_idx]
        
        # Category means
        cat_means = target_fold.groupby(train_fold).mean()
        cat_counts = train_fold.value_counts()
        
        # Smoothing: weight = count / (count + smoothing)
        smoothed_means = (cat_means * cat_counts + global_mean * smoothing) / (cat_counts + smoothing)
        
        # Apply to validation fold
        train_encoded.iloc[val_idx] = train_series.iloc[val_idx].map(smoothed_means).fillna(global_mean)
    
    # Encode test using full training data
    cat_means_full = target.groupby(train_series).mean()
    cat_counts_full = train_series.value_counts()
    smoothed_means_full = (cat_means_full * cat_counts_full + global_mean * smoothing) / (cat_counts_full + smoothing)
    test_encoded = test_series.map(smoothed_means_full).fillna(global_mean)
    
    return train_encoded.astype(np.float32), test_encoded.astype(np.float32)


# High-cardinality categoricals for target encoding
high_card_cols = [
    'ORGANIZATION_TYPE',  # 58 unique
    'OCCUPATION_TYPE',    # 18 unique
    'NAME_INCOME_TYPE',   # 8 unique
    'NAME_EDUCATION_TYPE', # 5 unique
    'NAME_FAMILY_STATUS', # 6 unique
    'NAME_HOUSING_TYPE',  # 6 unique
    'NAME_CONTRACT_TYPE'  # 2 unique
]

print("Target encoding function defined (will be applied inside CV folds)")
print(f"High-cardinality columns for target encoding: {high_card_cols}")

# Save high_card_cols for later use
with open(f'{WORK_DIR}/features/high_card_cols.json', 'w') as f:
    json.dump(high_card_cols, f)

In [ ]:
# 9. Adversarial Validation - Check Train/Test Distribution Shift
# If AUC > 0.6, there's significant distribution shift

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Prepare data for adversarial validation
# Use a subset of features for speed
adv_features = [c for c in train_fe.columns if c not in ['SK_ID_CURR'] and train_fe[c].dtype != 'object']
adv_features = adv_features[:50]  # Limit for speed

X_adv_train = train_fe[adv_features].fillna(-999)
X_adv_test = test_fe[adv_features].fillna(-999)

# Create adversarial target: 0=train, 1=test
y_adv_train = np.zeros(len(X_adv_train))
y_adv_test = np.ones(len(X_adv_test))

X_adv = pd.concat([X_adv_train, X_adv_test], axis=0)
y_adv = np.concatenate([y_adv_train, y_adv_test])

# Train adversarial classifier
X_adv_tr, X_adv_val, y_adv_tr, y_adv_val = train_test_split(
    X_adv, y_adv, test_size=0.3, random_state=42, stratify=y_adv
)

adv_clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=2)
adv_clf.fit(X_adv_tr, y_adv_tr)

adv_pred = adv_clf.predict_proba(X_adv_val)[:, 1]
adv_auc = roc_auc_score(y_adv_val, adv_pred)

print(f"Adversarial Validation AUC: {adv_auc:.4f}")
if adv_auc > 0.6:
    print("⚠️  WARNING: Significant train/test distribution shift detected!")
    print("    Consider: domain adaptation, feature selection, or pseudo-labeling")
else:
    print("✓ Train/test distributions are similar (AUC ~ 0.5)")

# Feature importance for adversarial model
adv_importance = pd.DataFrame({
    'feature': adv_features,
    'importance': adv_clf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 features distinguishing train vs test:")
print(adv_importance.head(10))

In [ ]:
# 10. Correlation Analysis - Identify Redundant Features
# High correlation features can be dropped to reduce noise

# Select numerical features only
num_features = train_fe.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numerical features: {len(num_features)}")

# Compute correlation matrix (sample for speed)
sample_size = min(10000, len(train_fe))
train_sample = train_fe[num_features].sample(n=sample_size, random_state=42)

corr_matrix = train_sample.corr()

# Find highly correlated pairs
high_corr_pairs = []
threshold = 0.95

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > threshold:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

print(f"\nHighly correlated pairs (|corr| > {threshold}): {len(high_corr_pairs)}")
for f1, f2, corr in high_corr_pairs[:20]:
    print(f"  {f1} <-> {f2}: {corr:.4f}")

# Save for feature selection later
high_corr_features = set()
for f1, f2, _ in high_corr_pairs:
    high_corr_features.add(f2)  # Drop second feature

with open(f'{WORK_DIR}/features/high_corr_features.json', 'w') as f:
    json.dump(list(high_corr_features), f)
print(f"\nFeatures to consider dropping: {len(high_corr_features)}")

In [ ]:
# 11. Final Preparation & Save Engineered Datasets

# Restore target and IDs
train_final = train_fe.copy()
train_final['TARGET'] = y.values
train_final['SK_ID_CURR'] = train_ids.values

test_final = test_fe.copy()
test_final['SK_ID_CURR'] = test_ids.values

# Feature list (excluding ID and target)
feature_cols = [c for c in train_final.columns if c not in ['SK_ID_CURR', 'TARGET']]
print(f"Final feature count: {len(feature_cols)}")
print(f"Train shape: {train_final.shape}")
print(f"Test shape: {test_final.shape}")

# Check for any remaining NaN
train_nan = train_final[feature_cols].isnull().sum().sum()
test_nan = test_final[feature_cols].isnull().sum().sum()
print(f"\nNaN in train features: {train_nan}")
print(f"NaN in test features: {test_nan}")

# Fill any remaining NaN with -999 (LightGBM/XGBoost handle this)
train_final[feature_cols] = train_final[feature_cols].fillna(-999)
test_final[feature_cols] = test_final[feature_cols].fillna(-999)

# Optimize dtypes for memory
for col in feature_cols:
    if train_final[col].dtype == 'float64':
        train_final[col] = train_final[col].astype(np.float32)
        test_final[col] = test_final[col].astype(np.float32)
    elif train_final[col].dtype == 'int64':
        # Check if can fit in int32
        if train_final[col].min() >= -2147483648 and train_final[col].max() <= 2147483647:
            train_final[col] = train_final[col].astype(np.int32)
            test_final[col] = test_final[col].astype(np.int32)

# Save as Parquet (fast, compressed)
train_final.to_parquet(f'{WORK_DIR}/features/train_engineered.parquet', index=False)
test_final.to_parquet(f'{WORK_DIR}/features/test_engineered.parquet', index=False)

# Save feature list
with open(f'{WORK_DIR}/features/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

# Save target
np.save(f'{WORK_DIR}/features/y_train.npy', y.values)

print("\n✓ Engineered datasets saved successfully!")
print(f"  Train: {WORK_DIR}/features/train_engineered.parquet")
print(f"  Test: {WORK_DIR}/features/test_engineered.parquet")
print(f"  Feature list: {WORK_DIR}/features/feature_cols.json")
print(f"  Target: {WORK_DIR}/features/y_train.npy")
print(f"  Label encoders: {WORK_DIR}/features/label_encoders.pkl")
print(f"  High card cols: {WORK_DIR}/features/high_card_cols.json")
print(f"  High corr features: {WORK_DIR}/features/high_corr_features.json")

In [ ]:
# 12. Memory Cleanup & Summary
gc.collect()

print("=== FEATURE ENGINEERING SUMMARY ===")
print(f"Original features: 33")
print(f"Engineered features: {len(feature_cols)}")
print(f"New features added: {len(feature_cols) - 33}")

print("\nFeature Categories:")
categories = {
    'Original': 33,
    'Missing Indicators': 8,
    'Domain Ratios': 13,
    'EXT_SOURCE Interactions': 13,
    'Bureau Aggregations': 5,
    'Quantile Bins': '~20',
    'Frequency Encodings': len(cat_cols),
    'Anomaly Indicators': 1
}
for cat, count in categories.items():
    print(f"  {cat}: {count}")

print("\n✅ Notebook 1 Complete! Ready for Notebook 2: Optuna Hyperparameter Tuning")
print()
print("Next steps:")
for step in [
    "02_optuna_tuning.ipynb is historical (its params were superseded by the E2 re-tune)",
    "03_ensemble_training_v2.ipynb is the historical ensemble runbook",
    "gci_honors_reproduction.ipynb is the self-contained submission artifact",
]:
    print("  -", step)
